# W07A Colab A · 데이터·평균·집단

딥러닝응용I(추천시스템) · 동덕여자대학교 데이터사이언스전공 · 유원상 교수 · 2026-2 · 2026년10월13일

각 노트북은 독립 실행됩니다. CPU에서 위부터 실행하세요. GPU·유료 API·파일 업로드가 필요 없습니다. 파일 → Drive에 사본 저장으로 자신의 실습본을 만드세요. 설치 후 재시작 안내가 나오면 재시작하고 첫 셀부터 다시 실행합니다.

설명용 데이터와 연습문제 데이터는 별개입니다. 문제 코드는 읽고 수정할 대상으로 markdown에 제시합니다. 자신의 답 문자열이나 코드가 비어 있어도 자동 시연과 검증은 독립 실행됩니다. 전체 실행 성공은 자신의 풀이가 정답이라는 뜻이 아닙니다.


## 코드 정의와 입력·출력

외부에서 가져오는 함수의 이름을 아래에서 클릭하면 설치한 고정 버전의 실제 정의 행을 볼 수 있습니다. 함수의 docstring을 먼저 읽고 입력과 반환값을 대조하세요. 함수 호출은 편의를 위한 것이며 Q40–Q60은 기본 Python만으로 풉니다.

| 함수 | 입력 | 출력·의미 |
|---|---|---|
| review_data | 없음 | train9행/test3행, 집단·장르를 담은 새 dict; 합성 표 자동 생성 |
| rating_stats | (사용자,영화,평점) 리스트, 선택 groups dict | 영화 또는 (집단,영화)별 sum/count/mean; 중복·잘못된 평점 거부 |
| mean_predict | train, 영화ID, 선택 집단, min_count | (평점,근거); 집단→영화→학습전체; test는 입력하지 않음 |
| cosine_score | 같은 길이의 두 수치 리스트 | 코사인 또는 영벡터일 때 None; 특징 순서 일치 필요 |
| rank_candidates | data dict, method/group/min_count/top_k | 미관람 후보를 조건 적용 후 정렬한 dict 목록 |
| rating_metrics | 같은 순서의 실제·예측 평점 리스트 | MAE/RMSE dict; 평가 관측 수로 나눔 |
| ranking_metrics | 추천ID·관련집합·카탈로그, k | P/R/NDCG/coverage; 짧은 목록의 P 분모도 요청K |
| review_experiment | 방법·최소수·K·집단 | 고정 합성 실험 결과 dict, 예측표·목록·지표 |
| review_view | 방법·최소수·K·집단, 선택 report_fn | 설명·추천표·예측표·지표표·그림; 실제 앱 callback |
| build_review_app | 선택 report_fn(result)->문자열 | Gradio 앱; 생성과 launch는 별도 |
| definition_link | 수업 함수/클래스/메소드, 고정ref | GitHub 정의 시작 행 URL; 파일·네트워크 변경 없음 |

이 함수들은 숨은 학습이나 파일 다운로드를 하지 않습니다. 핵심 데이터는 코드에 내장된 합성 표이며 입력과 출력에 명시된 통계만 계산합니다. 빈 학습 목록은 평균을 꾸미지 않고 오류로 보고합니다. 집단 목록 전체 대체와 쌍별 평점 대체의 적용 단위를 구별하세요.


In [ ]:
import os,sys,subprocess,math
os.environ['GRADIO_ANALYTICS_ENABLED']='False'
IN_COLAB='google.colab' in sys.modules
COURSE_VERSION='2026-fall-w07a'
if IN_COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q',
        f'luna-recommender-course[apps] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.26.0'],check=True)
from IPython.display import display,Markdown
from luna_recsys.review_foundations import (review_data,rating_stats,mean_predict,
    cosine_score,rank_candidates,rating_metrics,ranking_metrics,review_experiment)
from luna_recsys.review_app import review_view,build_review_app
from luna_recsys.model_comparison import definition_link
print('Python',sys.version.split()[0], '수업 버전',COURSE_VERSION)


In [ ]:
apis=[review_data,rating_stats,mean_predict,cosine_score,rank_candidates,
      rating_metrics,ranking_metrics,review_experiment,review_view,build_review_app,definition_link]
display(Markdown('\n'.join(f'- [{f.__name__}]({definition_link(f,COURSE_VERSION)})' for f in apis)))


## 1. 원자료를 먼저 읽기

아래 합성 표는 처음 실행할 때 자동 생성됩니다. 각 행은 (사용자,영화,평점)이며 train은 통계, test는 평가 전용입니다. 사용자u1/u2는G1, u3/u4는G2입니다. 영화 특징 순서는 [액션,코미디]입니다. 데이터를 눈으로 읽은 뒤 계산하세요.


In [ ]:
demo=review_data()
print('train',demo['train'])
print('test',demo['test'])
print('groups',demo['groups'])
print('features',demo['features'])


## 2. 합·개수·평균을 따로 보기

`rating_stats`는 같은 영화의 학습 관측만 모읍니다. 합과 개수를 먼저 읽으면 평균의 분모를 확인할 수 있습니다. 영화B는 서로 다른 사용자2명의 점수4와1을 가지므로 합5·평균2.5입니다. 이 시연값은 아래 연습문항과 별개입니다.


In [ ]:
demo_stats=rating_stats(demo['train'])
for item,stat in demo_stats.items():print(item,stat)
print('G1 movie B:',rating_stats(demo['train'],demo['groups'])[('G1','B')])


## 3. 대체 경로와 후보 순서

G1×B는 관측1개입니다. 최소수가1이면 집단 평균을, 2이면 영화 평균을 사용합니다. 영화Z에는 학습 관측이 없어 전체 평균으로 대체합니다. 예측값만 보지 말고 함께 반환한 근거를 읽으세요.


In [ ]:
for minimum in [1,2]:
    print(minimum,mean_predict(demo['train'],'B',groups=demo['groups'],group='G1',min_count=minimum))
print('new movie',mean_predict(demo['train'],'Z'))
for minimum in [1,2,4]:
    print('min_count=',minimum,rank_candidates(demo,min_count=minimum))


### LAB-A · 계산 근거 출력

목표: 평균 한 숫자 대신 합·개수·평균의 관계를 설명합니다. 영화B의 통계에서 sum/count/mean을 읽어 한 문장으로 출력하는 함수를 작성하세요. 힌트: `stats["B"]["count"]`는 B의 관측 수입니다. 자가 점검: 평균×개수와 합이 일치하는지 확인하세요.


In [ ]:
lab_a_reason=''  # 자신의 설명을 작성하세요. 자동 시연과 독립된 답 변수입니다.


## 4. 중간고사 대비 Python 연습

문제 번호는 학생 Notion·PDF와 같습니다. 먼저 출력이나 수정 방향을 종이에 적고 자신의 코드 셀에서 확인하세요. 문제의 잘못된 코드는 자동으로 실행하지 않습니다. 문법 설명은 각 문제에 제공되어 있습니다.


### Q40 · 개인 후보의 빈칸 (빈칸 / 기초)

빈칸 ___을 채워 이미 본 영화는 제외하라. `not in`은 목록에 포함되지 않음을 뜻한다. 결과도 쓰라.

```python
items=['A','B','C']
seen=['B']
out=[]
for item in items:
    if item ___ seen:
        out.append(item)
print(out)
```

목표: 추천 개념의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q40_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q41 · 클릭 기록의 논리 오류 (오류 / 기초)

코드는 클릭한 영화만 추천 후보로 남긴다. 이번 목표는 클릭하지 않은 영화를 탐색 후보로 만드는 것이다. 조건을 고치고, 비클릭을 싫음으로 단정할 수 없는 이유를 쓰라.

```python
clicked={'A':True,'B':False,'C':False}
out=[]
for item in clicked:
    if clicked[item]:
        out.append(item)
print(out)
```

목표: 추천 개념의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q41_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q42 · 관측 평점 수 (빈칸 / 기초)

`None`은 미관측이다. `is not None`은 실제 값이 있음을 검사한다. ___을 채워 관측 개수를 구하라.

```python
ratings=[5,None,2,None,4]
n=0
for r in ratings:
    if r is not None:
        n += ___
print(n)
```

목표: 데이터의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q42_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q43 · None을 포함한 평균 오류 (오류 / 기초)

다음 코드는 TypeError가 난다. 관측값만 모은 뒤 평균을 내도록 고치고 결과를 쓰라. 빈 목록은 없다고 가정한다.

```python
ratings=[5,None,1]
mean=sum(ratings)/len(ratings)
print(mean)
```

목표: 데이터의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q43_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q44 · ID 연결의 빈칸 (빈칸 / 기초)

딕셔너리는 키로 값을 찾는다. ___을 채워 각 평점에 맞는 집단을 출력하라.

```python
groups={'u2':'G2','u1':'G1'}
rows=[('u1','A',4),('u2','B',2)]
out=[]
for u,item,r in rows:
    out.append((item,groups[___],r))
print(out)
```

목표: 데이터의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q44_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q45 · 서로 다른 사용자 세기 (오류 / 적용)

출력은 평점 행 수를 세고 있다. 리스트만 사용하여 서로 다른 사용자 수를 출력하도록 고쳐라. `in`은 포함 여부를 검사한다.

```python
rows=[('u1','A',5),('u1','B',2),('u2','A',4)]
print(len(rows))
```

목표: 데이터의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q45_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q46 · 평균의 분모 (빈칸 / 기초)

sum은 합, len은 원소 개수이다. ___을 채워 평균을 계산하라.

```python
values=[2,4,3]
mean=sum(values)/___
print(mean)
```

목표: 인기·평균·후보의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q46_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q47 · 최소 개수 경계 오류 (오류 / 적용)

최소 평점 수2는2개도 포함한다. 다음 코드의 한 연산자를 고치고 결과를 쓰라.

```python
counts={'A':1,'B':2,'C':4}
out=[]
for item in counts:
    if counts[item] > 2:
        out.append(item)
print(out)
```

목표: 인기·평균·후보의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q47_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q48 · 평점 누적의 빈칸 (빈칸 / 적용)

각 행은 영화와 평점이다. ___을 채워 영화별 합을 누적하라. 딕셔너리에서 totals[item]은 그 영화의 현재 합이다.

```python
rows=[('A',2),('B',5),('A',4)]
totals={}
for item,r in rows:
    if item not in totals:
        totals[item]=0
    totals[item] += ___
print(totals)
```

목표: 인기·평균·후보의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q48_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q49 · 정렬 방향 오류 (오류 / 적용)

`sorted(scores,key=scores.get)`은 점수 오름차순이다. 낮은 점수를 추천하는 오류를 수정해 Top-2를 구하라. `reverse=True`는 내림차순이며 `[:2]`는 앞2개이다.

```python
scores={'A':2,'B':5,'C':4}
ranked=sorted(scores,key=scores.get)
print(ranked[:2])
```

목표: 인기·평균·후보의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q49_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q50 · 두 키로 집계하기 (빈칸 / 적용)

___에 (group,item) 튜플을 사용하도록 채워 집단×영화별 평점 목록을 만들라. 튜플은 두 값을 묶어 하나의 키로 사용한다.

```python
rows=[('G1','A',5),('G2','A',1),('G1','A',3)]
buckets={}
for group,item,r in rows:
    key=___
    if key not in buckets:
        buckets[key]=[]
    buckets[key].append(r)
print(buckets)
```

목표: 집단·분모·대체의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q50_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q51 · 집단 분모 오류 (오류 / 적용)

집단10명 중2명만 영화에 평점을 남겼다. 관측평균으로 고치고 분모의 의미를 쓰라.

```python
group_size=10
observed=[5,3]
mean=sum(observed)/group_size
print(mean)
```

목표: 집단·분모·대체의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q51_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q52 · 대체 경로의 빈칸 (빈칸 / 적용)

집단 관측 수가2 미만이면 영화 평균을 사용한다. ___을 채우고 출력하라. 영화 관측은 존재한다고 가정한다.

```python
group_values=[5]
movie_values=[5,3,1]
if len(group_values)>=2:
    prediction=sum(group_values)/len(group_values)
else:
    prediction=sum(___)/len(___)
print(prediction)
```

목표: 집단·분모·대체의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q52_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q53 · fallback 출력 추적 (출력 / 적용)

코드를 실행하지 않고 pred와 level의 최종 출력 및 실행한 분기를 쓰라. 딕셔너리의 `in`은 키 존재 여부이다.

```python
group_means={}
movie_means={'A':4}
global_mean=3
key=('G1','Z')
if key in group_means:
    pred=group_means[key];level='group'
elif key[1] in movie_means:
    pred=movie_means[key[1]];level='movie'
else:
    pred=global_mean;level='global'
print(pred,level)
```

목표: 집단·분모·대체의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q53_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


## 5. 실행 결과 검증

아래 검사는 실습의 핵심 기능과 평가 경계를 확인합니다. 자신의 연습문항 정답을 자동 채점하는 셀은 아닙니다. 오류가 나면 출력 내용을 확인하고 중단 원인을 해결하세요. 실패를 빈 결과로 바꾸지 않습니다.


In [ ]:
check_data=review_data()
assert not {(u,i) for u,i,_ in check_data['train']}&{(u,i) for u,i,_ in check_data['test']}
assert rating_stats(check_data['train'])['B']['mean']==2.5
assert mean_predict(check_data['train'],'B',groups=check_data['groups'],group='G1',min_count=2)==(2.5,'item mean')
assert [r['item'] for r in rank_candidates(check_data,min_count=2)]==['E','B']
assert cosine_score([0,0],[1,0]) is None
check_result=review_experiment('content',2,2,'G1')
assert 'RMSE' not in check_result['metrics']
assert len(check_result['ranked'])==2
print('W07A-A RUNTIME CHECK PASSED: train-only statistics, candidates, fallback, metrics.')


## 점검 퀴즈와 가져갈 생각

1. Q40를 다시 설명하세요: 개인 후보의 빈칸에서 핵심 조건은 무엇인가요?
2. Q41를 다시 설명하세요: 클릭 기록의 논리 오류에서 핵심 조건은 무엇인가요?
3. Q42를 다시 설명하세요: 관측 평점 수에서 핵심 조건은 무엇인가요?
4. Q43를 다시 설명하세요: None을 포함한 평균 오류에서 핵심 조건은 무엇인가요?
5. Q44를 다시 설명하세요: ID 연결의 빈칸에서 핵심 조건은 무엇인가요?

위 점검 질문은 이미 푼 동일 번호 문항을 다시 말로 설명하는 활동입니다. 새 문항이 아닙니다. 한 행의 의미, 관측 분모, 후보 처리 순서, 평가 정답의 분리를 기억하세요.

참고: 주교재1–2장, 기존W01–W02 수업. [Google 내용 기반](https://developers.google.com/machine-learning/recommendation/content-based/basics), [scikit-learn 누수](https://scikit-learn.org/stable/common_pitfalls.html), [NumPy SVD](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html).


다음 활동은 Colab B입니다. [내용·평가·마지막 앱 열기](https://colab.research.google.com/github/lunalab-ai/recommender/blob/2026-fall-w07a/notebooks/student/w07a-review-content-app.ipynb). 이 노트북을 실행한 기록 없이도 B를 독립 실행할 수 있습니다.
